# Eksperimen, Preprocessing & Feature Engineering
Notebook ini didedikasikan untuk memvisualisasikan seluruh proses di balik layar secara *step-by-step*, sesuai dengan panduan proyek:
- Distribusi Data setelah Split
- Hasil Data Augmentasi
- Proses Preprocessing (Resizing, Grayscale, CLAHE, Blur, Thresholding/Morfologi)
- Ekstraksi Fitur (Canny, DWT, Flatten, Scaling)

In [ ]:
import sys
import os
# Jika script diupload sebagai dataset Kaggle, hilangkan tanda pagar di bawah ini dan sesuaikan foldernya
sys.path.append('/kaggle/input/datasets/emageeeee/pcd-k23')

import cv2
import pywt
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from config import CLASSES, IMG_SIZE_CNN
from src.preprocess import *

kaggle_dir = download_data_from_kaggle()
remove_blank_images(kaggle_dir)
remove_duplicate_images(kaggle_dir)
print("Data raw terhubung di:", kaggle_dir)

## 1. Splitting Dataset & Distribusi Data

In [ ]:
def show_split_distribution():
    splits = ['Train', 'Validation', 'Test']
    counts = []
    for split in splits:
        for cls in CLASSES:
            path = kaggle_dir / split / cls
            n_images = len(list(path.glob("*.png")) + list(path.glob("*.jpg")))
            counts.append({'Split': split, 'Class': cls, 'Total': n_images})
            
    df = pd.DataFrame(counts)
    display(df.pivot(index='Split', columns='Class', values='Total'))
    
    plt.figure(figsize=(8,4))
    sns.barplot(data=df, x='Split', y='Total', hue='Class')
    plt.title("Distribusi Data Setelah Splitting")
    plt.show()

show_split_distribution()

## 2. Preprocessing (Step-by-Step)

In [ ]:
sample_path = list((kaggle_dir / 'Train' / CLASSES[0]).glob("*.png"))[0]
img_orig = cv2.imread(str(sample_path))
img_rgb = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)

def plot_comparison(img1, img2, title1, title2, cmap1=None, cmap2=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img1, cmap=cmap1)
    axes[0].set_title(title1)
    axes[0].axis('off')
    
    axes[1].imshow(img2, cmap=cmap2)
    axes[1].set_title(title2)
    axes[1].axis('off')
    plt.show()

### 2.1 Resizing (Transformasi Linear)

In [ ]:
img_resized = cv2.resize(img_rgb, IMG_SIZE_CNN)
plot_comparison(img_rgb, img_resized, f"Original: {img_rgb.shape}", f"Resized: {img_resized.shape}")

### 2.2 Grayscale Conversion

In [ ]:
img_gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)
plot_comparison(img_resized, img_gray, "RGB", "Grayscale", cmap2='gray')

### 2.3 Histogram Equalization (CLAHE)

In [ ]:
# clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
img_clahe = clahe_scratch(img_gray)
plot_comparison(img_gray, img_clahe, "Grayscale Biasa", "Setelah CLAHE", cmap1='gray', cmap2='gray')

### 2.4 Noise Filtering (Gaussian Blur)

In [ ]:
img_blur = gaussian_blur_scratch(img_clahe)
plot_comparison(img_clahe, img_blur, "Sebelum Blur", "Sesudah Gaussian Blur", cmap1='gray', cmap2='gray')

### 2.5 Sharpening Filtering

In [ ]:
img_sharp = sharpen_scratch(img_blur)
plot_comparison(img_blur, img_sharp, "Sebelum Sharpening Filter", "Sesudah", cmap1='gray', cmap2='gray')

### 2.6 Tresholding Adaptive

In [ ]:
img_thresh = adaptive_threshold_scratch(img_sharp)
plot_comparison(img_sharp, img_thresh, "Sebelum ", "Sesudah", cmap1='gray', cmap2='gray')

### 2.7 Morfologi Citra (Thresholding & Morphological Opening/Closing)
*(Eksperimen Tambahan sesuai catatan notes.txt)*

In [ ]:
_, img_thresh = cv2.threshold(img_blur, 127, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
kernel = np.ones((3,3), np.uint8)
img_opening = cv2.morphologyEx(img_thresh, cv2.MORPH_OPEN, kernel)
img_closing = cv2.morphologyEx(img_opening, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_thresh, cmap='gray')
axes[0].set_title("Thresholding (Otsu)")
axes[0].axis('off')

axes[1].imshow(img_opening, cmap='gray')
axes[1].set_title("Morphological Opening")
axes[1].axis('off')

axes[2].imshow(img_closing, cmap='gray')
axes[2].set_title("Morphological Closing")
axes[2].axis('off')
plt.show()

## 3. Ekstraksi Fitur (ML Klasik)

### 3.1 Edge Detection (Canny)

In [ ]:
img_canny = cv2.Canny(img_thresh, 100, 200)
plot_comparison(img_thresh, img_canny, "Gambar Preprocessed (Blur)", "Fitur Canny Edge", cmap1='gray', cmap2='gray')

### 3.2 Transformasi Wavelet (DWT)

In [ ]:
coeffs = pywt.dwt2(img_thresh, 'haar')
LL, (LH, HL, HH) = coeffs

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(LL, cmap='gray')
axes[0, 0].set_title("LL (Approximation)")
axes[0, 0].axis('off')

axes[0, 1].imshow(LH, cmap='gray')
axes[0, 1].set_title("LH (Horizontal Detail)")
axes[0, 1].axis('off')

axes[1, 0].imshow(HL, cmap='gray')
axes[1, 0].set_title("HL (Vertical Detail)")
axes[1, 0].axis('off')

axes[1, 1].imshow(HH, cmap='gray')
axes[1, 1].set_title("HH (Diagonal Detail)")
axes[1, 1].axis('off')
plt.show()

### 3.3 Flattening & Feature Scaling
Tahap akhir di mana gambar 2D diubah menjadi deretan angka 1D dan dinormalisasi untuk SVM.

In [ ]:
from sklearn.preprocessing import StandardScaler

flatten_canny = img_canny.flatten()
flatten_dwt = LL.flatten()

print("Shape asli Canny:", img_canny.shape)
print("Setelah Flatten (1D):", flatten_canny.shape)

print("\nShape asli DWT (LL):", LL.shape)
print("Setelah Flatten (1D):", flatten_dwt.shape)

# Simulasi StandardScaler (harus 2D input)
scaler = StandardScaler()
scaled_canny = scaler.fit_transform(flatten_canny.reshape(-1, 1))

print("\n5 Nilai Array Flatten (Sebelum Scaling):\n", flatten_canny[:5])
print("\n5 Nilai Array Flatten (Sesudah Scaling / Normalisasi):\n", scaled_canny[:5].ravel())

## 4. Audit ROI Ulang setelah Enhance Data

In [ ]:
sample_paths = []
for split in ['Train', 'Validation']:
    for label in ['WithMask', 'WithoutMask']:
        folder = kaggle_dir / split / label
        if folder.exists():
            images = list(folder.glob('*.png')) + list(folder.glob('*.jpg'))
            np.random.seed(42)
            np.random.shuffle(images)
            for p in images[:100]:
                sample_paths.append({'path': str(p), 'split': split, 'label': label})

sample_df = pd.DataFrame(sample_paths)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

audit_results = []
rescued_images = []

print(f"Mengaudit {len(sample_df)} gambar untuk membandingkan ROI Sebelum vs Sesudah Enhancement...\n")

for _, row in sample_df.iterrows():
    img_bgr = cv2.imread(row['path'])
    if img_bgr is None: continue
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    
    faces_before = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(24, 24))
    detected_before = int(len(faces_before) > 0)
    
    img_clahe = clahe_scratch(gray)
    img_blur = gaussian_blur_scratch(img_clahe)
    img_sharp = sharpen_scratch(img_blur)
    
    faces_after = face_cascade.detectMultiScale(img_sharp, scaleFactor=1.1, minNeighbors=5, minSize=(24, 24))
    detected_after = int(len(faces_after) > 0)
    
    audit_results.append({
        'split': row['split'],
        'label': row['label'],
        'detected_before': detected_before,
        'detected_after': detected_after
    })
    
    if detected_before == 0 and detected_after == 1:
        img_sharp_bgr = cv2.cvtColor(img_sharp, cv2.COLOR_GRAY2BGR)
        for (x, y, w, h) in faces_after:
            cv2.rectangle(img_sharp_bgr, (x, y), (x+w, y+h), (0, 255, 0), 2)
            
        rescued_images.append({
            'label': row['label'],
            'original': cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB),
            'enhanced': cv2.cvtColor(img_sharp_bgr, cv2.COLOR_BGR2RGB)
        })

audit_df = pd.DataFrame(audit_results)
summary = audit_df.groupby(['split', 'label'])[['detected_before', 'detected_after']].sum()
summary['peningkatan'] = summary['detected_after'] - summary['detected_before']
display(summary)

print(f"\n[KESIMPULAN UMUM]")
print(f"Total wajah terdeteksi SEBELUM enhancement: {audit_df['detected_before'].sum()} / {len(audit_df)}")
print(f"Total wajah terdeteksi SESUDAH enhancement: {audit_df['detected_after'].sum()} / {len(audit_df)}")
print(f"Enhancement menyelamatkan {len(rescued_images)} gambar buta-fitur menjadi bisa terdeteksi!")

n_show = min(5, len(rescued_images))
if n_show > 0:
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    if n_show == 1: axes = np.array([axes]).T 
    
    for i in range(n_show):
        data = rescued_images[i]
        axes[0, i].imshow(data['original'])
        axes[0, i].set_title(f"{data['label']}\nSEBELUM\n(Gagal Terdeteksi)", color='red')
        axes[0, i].axis('off')
        
        axes[1, i].imshow(data['enhanced'])
        axes[1, i].set_title(f"SESUDAH\n(Berhasil Terdeteksi!)", color='green')
        axes[1, i].axis('off')
        
    plt.tight_layout()
    plt.show()
